## Palm Movement Detection using Optical Flow

This lab is part of [AI for Beginners Curriculum](http://aka.ms/ai-beginners).

Consider [this video](palm-movement.mp4), in which a person's palm moves left/right/up/down on the stable background.

<img src="../images/palm-movement.png" width="30%" alt="Palm Movement Frame"/>

**Your goal** would be to use Optical Flow to determine, which parts of video contain up/down/left/right movements. 

Start by getting video frames as described in the lecture:

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Load video frames
vid = cv2.VideoCapture('/home/silent/base/AI-For-Beginners/lessons/4-ComputerVision/06-IntroCV/lab/palm-movement.mp4')

frames = []
while vid.isOpened():
    ret, frame = vid.read()
    if not ret:
        break
    frames.append(frame)
vid.release()
print(f"Total frames: {len(frames)}")

# Display some frames
def display_images(l, titles=None, fontsize=12):
    n = len(l)
    fig, ax = plt.subplots(1, n, figsize=(15, 5))
    if n == 1:
        ax = [ax]
    for i, im in enumerate(l):
        ax[i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax[i].axis('off')
        if titles is not None:
            ax[i].set_title(titles[i], fontsize=fontsize)
    plt.tight_layout()
    plt.show()

display_images(frames[::30], titles=[f"Frame {i*30}" for i in range(len(frames[::30]))])

Now, calculate dense optical flow frames as described in the lecture, and convert dense optical flow to polar coordinates: 

In [ ]:
# Convert frames to grayscale
bwframes = [cv2.cvtColor(x, cv2.COLOR_BGR2GRAY) for x in frames]

# Calculate dense optical flow between consecutive frames
flows = []
for i in range(len(bwframes) - 1):
    flow = cv2.calcOpticalFlowFarneback(bwframes[i], bwframes[i+1], None, 0.5, 3, 15, 3, 5, 1.2, 0)
    flows.append(flow)

# Convert optical flow to polar coordinates (magnitude and angle)
def flow_to_polar(flow):
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    return mag, ang

# Calculate polar coordinates for all flows
polar_flows = [flow_to_polar(f) for f in flows]

print(f"Number of optical flow frames: {len(flows)}")
print(f"Shape of flow: {flows[0].shape}")

Build histogram of directions for each of the optical flow frame. A histogram shows how many vectors fall under certain bin, and it should separate out different directions of movement on the frame.

> You may also want to zero out all vectors whose magnitude is below certain threshold. This will get rid of small extra movements in the video, such as eyes and head.

Plot the histograms for some of the frames.

In [ ]:
# Build histogram of directions for each optical flow frame
# Angles are in radians, range [0, 2π]
# We'll use 4 bins for the 4 main directions: right, up, left, down

def build_direction_histogram(mag, ang, mag_threshold=1.0):
    """
    Build histogram of movement directions.
    Angles: 0 = right, π/2 = up, π = left, 3π/2 = down
    """
    # Zero out small magnitude vectors
    mask = mag > mag_threshold
    angles = ang[mask]
    
    # Create histogram with 4 bins for main directions
    # Bin 0: right (angles around 0) - angles [7π/4, 2π) and [0, π/4)
    # Bin 1: up (angles around π/2) - angles [π/4, 3π/4)
    # Bin 2: left (angles around π) - angles [3π/4, 5π/4)
    # Bin 3: down (angles around 3π/2) - angles [5π/4, 7π/4)
    
    hist, _ = np.histogram(angles, bins=8, range=(0, 2*np.pi))
    return hist

# Build histograms for all frames
histograms = []
for mag, ang in polar_flows:
    hist = build_direction_histogram(mag, ang, mag_threshold=1.0)
    histograms.append(hist)

histograms = np.array(histograms)
print(f"Histograms shape: {histograms.shape}")

# Plot histograms for selected frames
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
frame_indices = [0, 20, 40, 60, 80, 100, 120, 140]
directions = ['→', '↗', '↑', '↖', '←', '↙', '↓', '↘']

for idx, ax in zip(frame_indices, axes.flat):
    if idx < len(histograms):
        ax.bar(range(8), histograms[idx])
        ax.set_title(f"Frame {idx}")
        ax.set_xticks(range(8))
        ax.set_xticklabels(directions, fontsize=10)
        ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

# Also plot the histogram over time as a heatmap
plt.figure(figsize=(12, 6))
plt.imshow(histograms.T, aspect='auto', cmap='hot')
plt.colorbar(label='Count')
plt.yticks(range(8), directions)
plt.xlabel('Frame')
plt.ylabel('Direction')
plt.title('Movement Direction Histograms Over Time')
plt.show()

Looking at histograms, it should be pretty straightforward how to determine direction of movement. You need so select those bins the correspond to up/down/left/right directions, and that are above certain threshold.

In [ ]:
# Determine movement direction for each frame
# We use 4 main directions: right (0), up (1), left (2), down (3)
# Map 8-bin histogram to 4 directions by combining adjacent bins

def get_movement_direction(hist, threshold=500):
    """
    Determine the dominant movement direction from histogram.
    Returns: 'right', 'left', 'up', 'down', or 'none'
    """
    # Combine 8 bins into 4 main directions
    # right: bins 0, 7 (angles around 0)
    # up: bins 1, 2 (angles around π/2)
    # left: bins 3, 4 (angles around π)
    # down: bins 5, 6 (angles around 3π/2)
    
    right = hist[0] + hist[7]
    up = hist[1] + hist[2]
    left = hist[3] + hist[4]
    down = hist[5] + hist[6]
    
    directions = {
        'right': right,
        'up': up,
        'left': left,
        'down': down
    }
    
    # Find the dominant direction
    max_dir = max(directions, key=directions.get)
    max_val = directions[max_dir]
    
    if max_val > threshold:
        return max_dir
    return 'none'

# Determine direction for each frame
frame_directions = []
for hist in histograms:
    direction = get_movement_direction(hist, threshold=500)
    frame_directions.append(direction)

# Print segments of movement
print("Movement direction per frame:")
current_dir = frame_directions[0]
start_frame = 0

for i, d in enumerate(frame_directions):
    if d != current_dir:
        if current_dir != 'none':
            print(f"Frames {start_frame}-{i-1}: {current_dir.upper()}")
        current_dir = d
        start_frame = i

if current_dir != 'none':
    print(f"Frames {start_frame}-{len(frame_directions)-1}: {current_dir.upper()}")

# Create a visualization of movement directions over time
direction_colors = {
    'right': 'green',
    'left': 'blue', 
    'up': 'red',
    'down': 'orange',
    'none': 'gray'
}

plt.figure(figsize=(14, 5))
colors = [direction_colors[d] for d in frame_directions]
plt.bar(range(len(frame_directions)), [1]*len(frame_directions), color=colors)
plt.title('Movement Direction Over Time')
plt.xlabel('Frame')
plt.yticks([])

# Create legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=d) for d, c in direction_colors.items()]
plt.legend(handles=legend_elements, loc='upper right')
plt.show()

print("\nDirection legend:")
print("Green = Right, Blue = Left, Red = Up, Orange = Down, Gray = None")

Congratulations! If you have done all steps above, you have completed the lab!